# Pipeline Medallion - Control de Inventario
## Proyecto: Análisis de Provisión y Valor de Inventario

**Objetivo:** Implementar arquitectura medallion (Bronze → Silver → Gold) para análisis temporal de inventario por:
- Unidad de negocio
- Producto
- Categoría de provisión

**Fuente de datos:** `/Volumes/workshop_aleja/default/trabajo_final/Inventario_databricks.xlsx`

**Capas:**
- 🟫 **Bronze**: Datos crudos sin transformaciones
- ⚪ **Silver**: Datos limpios, validados y normalizados
- 🟡 **Gold**: Métricas agregadas listas para análisis y dashboards

## PASO 1: Exploración del Archivo Excel

Antes de construir el pipeline, exploramos la estructura del archivo para identificar:
- Nombres exactos de las columnas
- Tipos de datos
- Cantidad de registros
- Valores de ejemplo

In [0]:
# Exploración del archivo Excel de inventario
from pyspark.sql.functions import *

# Ruta del archivo Excel en el volumen
excel_path = "/Volumes/workshop_aleja/default/trabajo_final/Inventario_databricks.xlsx"

print("📂 Leyendo archivo Excel...")
print(f"Ruta: {excel_path}\n")

# Leer el archivo Excel con inferencia automática de esquema
df_inventario_raw = (
    spark.read
    .format("excel")
    .option("headerRows", 1)  # Primera fila como encabezado
    .option("inferSchema", True)  # Inferir tipos de datos automáticamente
    .load(excel_path)
)

# Mostrar estructura del archivo
print("=" * 80)
print("📋 ESTRUCTURA DEL ARCHIVO - COLUMNAS Y TIPOS DE DATOS")
print("=" * 80)
df_inventario_raw.printSchema()

print("\n" + "=" * 80)
print(f"📊 TOTAL DE REGISTROS: {df_inventario_raw.count():,}")
print(f"📊 TOTAL DE COLUMNAS: {len(df_inventario_raw.columns)}")
print("=" * 80)

print("\n📝 NOMBRES DE COLUMNAS:")
for i, col_name in enumerate(df_inventario_raw.columns, 1):
    print(f"  {i}. {col_name}")

print("\n" + "=" * 80)
print("👀 MUESTRA DE DATOS (primeras 10 filas)")
print("=" * 80)

📂 Leyendo archivo Excel...
Ruta: /Volumes/workshop_aleja/default/trabajo_final/Inventario_databricks.xlsx

📋 ESTRUCTURA DEL ARCHIVO - COLUMNAS Y TIPOS DE DATOS
root
 |-- Item: string (nullable = true)
 |-- Unidad de negocio: string (nullable = true)
 |-- Descripción Item: string (nullable = true)
 |-- Fecha: timestamp_ntz (nullable = true)
 |-- UM: string (nullable = true)
 |-- Tipo de Ítem: string (nullable = true)
 |-- Grupo Inventario: string (nullable = true)
 |-- Entradas Últ. 12 Meses: decimal(33,13) (nullable = true)
 |-- Salidas Últ. 12 Meses: decimal(35,15) (nullable = true)
 |-- Entradas Últ. 24 Meses: decimal(34,14) (nullable = true)
 |-- Salidas Últ. 24 Meses: decimal(36,16) (nullable = true)
 |-- Entradas entre\n12 y 24 Meses: decimal(33,13) (nullable = true)
 |-- Salidas entre\n12 y 24 Meses: decimal(37,17) (nullable = true)
 |-- Entrada 12 meses y saldo: string (nullable = true)
 |-- % entrada 12 meses / Saldo: double (nullable = true)
 |-- Saldo: string (nullable = true

In [0]:
# Visualizar muestra de datos
display(df_inventario_raw.limit(10))

Item,Unidad de negocio,Descripción Item,Fecha,UM,Tipo de Ítem,Grupo Inventario,Entradas Últ. 12 Meses,Salidas Últ. 12 Meses,Entradas Últ. 24 Meses,Salidas Últ. 24 Meses,Entradas entre 12 y 24 Meses,Salidas entre 12 y 24 Meses,Entrada 12 meses y saldo,% entrada 12 meses / Saldo,Saldo,Valor STD (COP$),Valor Real (COP$),Cant. Backlog,Valor Backlog (R$),Backlog ajustado cant. (de acuerdo a inventario existente,Backlog ajustado valor (de acuerdo a inventario existente,Cantidad en contratos,Inventario cubierto por contratos COP$,Total inventario no comprometido después de backlog,Total inventario no comprometido después de backlog y contratos,Agotamiento,Cant. Comprometida,Nuevo agotamiento,%,Categoría Provisión de Inventario,Valor Provisión
00030086,PR,Producto 00030086,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,5146.0000000000000,0E-15,12256.00000000000000,1389.0000000000000000,7110.0000000000000,1389.00000000000000000,,0.45774773172033445,"11,242.00",167894329.02860001000000,167894329.02860001000000,0,0E-14,0E-14,0E-14,null,0,1.678943290286E8,1.678943290286E8,0E-16,1656.082000000000100000,999.9,0.0,C,"83,947,164.5"
00030102,PR,Producto 00030102,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,9074.0000000000000,2295.393999999999800,9074.00000000000000,10743.7040000000000000,0E-13,8448.31000000000000000,(966),1.119141588554514,"8,108.00",119330971.41270000000000,119330971.41270000000000,0,0E-14,0E-14,0E-14,null,0,1.193309714127E8,1.193309714127E8,42.4000000000000000,6.499699999999999800,30.387494260244647,0.28310236803157374,Regular,- 0
00030121,PR,Producto 00030121,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,0E-13,6536.800000000000000,23912.00000000000000,16047.5120000000010000,23912.0000000000000,9510.71199999999950000,,0.0,"29,025.20",438873014.60039997000000,438873014.60039997000000,0,0E-14,0E-14,0E-14,null,0,4.388730146004E8,4.388730146004E8,52.7000000000000000,296.240000000000000000,41.283319055195207,0.22521119578848725,B,"131,661,904.4"
00030136,PR,Producto 00030136,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,9686.0000000000000,2062.000000000000000,21924.00000000000000,4158.5180000000000000,12238.0000000000000,2096.51800000000000000,,0.5267565803785077,"18,388.00",260903353.02070001000000,260903353.02070001000000,0,0E-14,0E-14,0E-14,null,0,2.609033530207E8,2.609033530207E8,94.0000000000000000,2237.449999999999800000,95.010669253152273,0.11213835109854253,Regular,- 0
00030146,PR,Producto 00030146,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,4028.0000000000000,311.000000000000000,7016.00000000000000,2191.6000000000000000,2988.0000000000000,1880.60000000000000000,,0.4744405182567727,"8,490.00",120328307.46480000000000,120328307.46480000000000,0,0E-14,0E-14,0E-14,null,0,1.203283074648E8,1.203283074648E8,327.6000000000000200,0E-18,315.58842443729901,0.036631330977620724,B,"36,098,492.2"
00030146-PLPBR,PR,Producto 00030146-PLPBR,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,0E-13,565.000000000000000,0E-14,2387.0000000000000000,0E-13,1822.00000000000000000,443,0.0,443.00,13250079.57330000000000,13250079.57330000000000,0,0E-14,0E-14,0E-14,null,0,1.32500795733E7,1.32500795733E7,5.6000000000000000,178.425000000000010000,0,1.275395033860045,Regular,- 0
00030167,PR,Producto 00030167,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,7110.0000000000000,3725.304999999999800,11088.00000000000000,12868.8050000000000000,3978.0000000000000,9143.50000000000000000,,0.7590477207216825,"9,367.00",133733061.03160000000000,133733061.03160000000000,0,0E-14,0E-14,0E-14,null,0,1.337330610316E8,1.337330610316E8,26.4000000000000000,1183.005000000000100000,18.173099920677636,0.3977052418063415,Regular,- 0
00030182,PR,Producto 00030182,2026-04-30T00:00:00.000,KG,Materia Prima,MATERIA PRIMA,0E-13,3901.700000000000000,0E-14,3901.7000000000000000,0E-13,0E-17,205,0.0,205.00,3170315.53929999980000,3170315.53929999980000,0,0E-14,0E-14,0E-14,null,0,3170315.5393,3170315.5393,0.6000000000000000,0E-18

In [0]:
# Mostrar estadísticas descriptivas de columnas numéricas
print("📈 ESTADÍSTICAS DESCRIPTIVAS DE COLUMNAS NUMÉRICAS:\n")
df_inventario_raw.describe().display()

📈 ESTADÍSTICAS DESCRIPTIVAS DE COLUMNAS NUMÉRICAS:



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5309107989756666>, line 3
      1 # Mostrar estadísticas descriptivas de columnas numéricas
      2 print("📈 ESTADÍSTICAS DESCRIPTIVAS DE COLUMNAS NUMÉRICAS:\n")
----> 3 df_inventario_raw.describe().display()

File /databricks/python_shell/lib/dbruntime/monkey_patches.py:67, in apply_dataframe_display_patch.<locals>.df_display(df, *args, **kwargs)
     63 def df_display(df, *args, **kwargs):
     64     """
     65     df.display() is an alias for display(df). Run help(display) for more information.
     66     """
---> 67     display(df, *args, **kwargs)

File /databricks/python_shell/lib/dbruntime/display.py:142, in Display.display(self, input, *args, **kwargs)
    140 # This version is for Serverless + Spark Connect dogfooding.
    141 elif self.spark_connect_enabled and isinstance(input, ConnectDataFrame):
--> 142     

## 🟫 PASO 2: Capa BRONZE - Datos Crudos

La capa Bronze almacena los datos tal como vienen del Excel, **sin transformaciones**.

**Agregamos metadatos de auditoría:**
- `_fecha_ingesta`: timestamp de cuando se cargó el dato
- `_archivo_origen`: nombre del archivo fuente

**Objetivo:** Crear tabla `workshop_aleja.default.inventario_bronze`

In [0]:
# Crear tabla BRONZE con datos crudos y metadatos de auditoría
from pyspark.sql.functions import current_timestamp, lit, col, regexp_replace

print("🟫 Creando tabla BRONZE...\n")

# Función para normalizar nombres de columnas para Delta Lake
def normalizar_nombre_columna(nombre):
    """
    Normaliza nombres de columnas eliminando caracteres no permitidos en Delta.
    Delta no permite: espacios, comas, punto y coma, {}, (), \n, \t, =
    """
    import re
    # Convertir a minúsculas
    nombre = nombre.lower()
    # Reemplazar espacios, saltos de línea y tabs por guion bajo
    nombre = re.sub(r'[\s\n\r\t]+', '_', nombre)
    # Eliminar TODOS los paréntesis (abiertos, cerrados, y su contenido)
    nombre = re.sub(r'[\(\)]', '', nombre)
    # Eliminar llaves
    nombre = re.sub(r'[\{\}]', '', nombre)
    # Reemplazar caracteres especiales
    nombre = nombre.replace('.', '')
    nombre = nombre.replace(',', '')
    nombre = nombre.replace(';', '')
    nombre = nombre.replace('=', '')
    nombre = nombre.replace('/', '_')
    nombre = nombre.replace('%', 'pct')
    nombre = nombre.replace('$', '_dolares')
    nombre = nombre.replace('ñ', 'n')
    # Eliminar tildes y acentos
    reemplazos = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        'à': 'a', 'è': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u'
    }
    for viejo, nuevo in reemplazos.items():
        nombre = nombre.replace(viejo, nuevo)
    # Eliminar guiones bajos consecutivos
    nombre = re.sub(r'_+', '_', nombre)
    # Eliminar guiones bajos al inicio y final
    nombre = nombre.strip('_')
    return nombre

print("⚙️ Normalizando nombres de columnas para Delta Lake...\n")

# Normalizar nombres de columnas
df_bronze = df_inventario_raw
for nombre_original in df_bronze.columns:
    nombre_nuevo = normalizar_nombre_columna(nombre_original)
    if nombre_original != nombre_nuevo:
        df_bronze = df_bronze.withColumnRenamed(nombre_original, nombre_nuevo)
        # Solo mostrar algunos ejemplos para no saturar la salida
        if df_bronze.columns.index(nombre_nuevo) < 10:
            print(f"  ✓ '{nombre_original}' → '{nombre_nuevo}'")

print(f"\n✅ {len(df_bronze.columns) - len(df_inventario_raw.columns)} columnas renombradas")

# Agregar metadatos de auditoría
df_bronze = (
    df_bronze
    .withColumn("_fecha_ingesta", current_timestamp())
    .withColumn("_archivo_origen", lit("Inventario_databricks.xlsx"))
)

print(f"📊 Total de columnas: {len(df_bronze.columns)} (32 originales + 2 metadatos)")

# Guardar en tabla Bronze
print(f"\n💾 Guardando tabla Bronze en Delta Lake...")
tabla_bronze = "workshop_aleja.default.inventario_bronze"

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_bronze)

print(f"\n✅ Tabla BRONZE creada exitosamente: {tabla_bronze}")
print(f"📊 Registros cargados: {df_bronze.count():,}")
print(f"📅 Fecha de ingesta: {df_bronze.select('_fecha_ingesta').first()[0]}")

🟫 Creando tabla BRONZE...

⚙️ Normalizando nombres de columnas para Delta Lake...

  ✓ 'Item' → 'item'
  ✓ 'Unidad de negocio' → 'unidad_de_negocio'
  ✓ 'Descripción Item' → 'descripcion_item'
  ✓ 'Fecha' → 'fecha'
  ✓ 'UM' → 'um'
  ✓ 'Tipo de Ítem' → 'tipo_de_item'
  ✓ 'Grupo Inventario' → 'grupo_inventario'
  ✓ 'Entradas Últ. 12 Meses' → 'entradas_ult_12_meses'
  ✓ 'Salidas Últ. 12 Meses' → 'salidas_ult_12_meses'
  ✓ 'Entradas Últ. 24 Meses' → 'entradas_ult_24_meses'

✅ 0 columnas renombradas
📊 Total de columnas: 34 (32 originales + 2 metadatos)

💾 Guardando tabla Bronze en Delta Lake...

✅ Tabla BRONZE creada exitosamente: workshop_aleja.default.inventario_bronze
📊 Registros cargados: 22,071
📅 Fecha de ingesta: 2026-05-12 22:48:04.304112


In [0]:
# Visualizar resultado de tabla Bronze
print("👀 Muestra de la tabla BRONZE (primeras 5 filas):\n")
display(spark.table(tabla_bronze).limit(5))

## ⚪ PASO 3: Capa SILVER - Limpieza y Transformación

La capa Silver limpia y normaliza los datos de Bronze para facilitar el análisis.

**Transformaciones aplicadas:**
1. ✅ Convertir `saldo` de string a numérico (eliminar comas)
2. ✅ Convertir `valor_provision` de string a numérico (eliminar comas y paréntesis)
3. ✅ Eliminar registros con valores nulos en campos críticos
4. ✅ Crear columnas calculadas para análisis
5. ✅ Validar calidad de datos

**Objetivo:** Crear tabla `workshop_aleja.default.inventario_silver`

In [0]:
# Crear tabla SILVER con limpieza y transformaciones
from pyspark.sql.functions import (
    col, regexp_replace, when, trim, round as spark_round, 
    coalesce, lit, to_date, expr
)
from pyspark.sql.types import DecimalType, DoubleType

print("⚪ Creando tabla SILVER...\n")

# Leer desde Bronze
df_silver = spark.table("workshop_aleja.default.inventario_bronze")

print("⚙️ Aplicando transformaciones...\n")

# 1. Convertir 'saldo' de string a numérico (eliminar comas y convertir)
print("  1. Convertir 'saldo' a numérico")
df_silver = df_silver.withColumn(
    "saldo_limpio",
    regexp_replace(col("saldo"), ",", "")  # Eliminar comas
)
df_silver = df_silver.withColumn(
    "saldo_numerico",
    when(
        (trim(col("saldo_limpio")) == "") | (trim(col("saldo_limpio")).isNull()), 
        lit(None)
    ).otherwise(
        expr("try_cast(saldo_limpio as double)")  # try_cast retorna NULL si falla
    )
).drop("saldo_limpio")

# 2. Convertir 'valor_provision' a numérico (manejar formatos como "83,947,164.5" y "- 0")
print("  2. Convertir 'valor_provision' a numérico")
df_silver = df_silver.withColumn(
    "valor_provision_limpio",
    # Eliminar comas, espacios extra y guiones al inicio
    trim(regexp_replace(regexp_replace(col("valor_provision"), ",", ""), "^-\\s*", ""))
)
df_silver = df_silver.withColumn(
    "valor_provision_numerico",
    when(
        (col("valor_provision_limpio") == "0") | (col("valor_provision_limpio") == "") | (col("valor_provision_limpio").isNull()), 
        lit(0)
    ).otherwise(
        expr("try_cast(valor_provision_limpio as double)")  # try_cast retorna NULL si falla
    )
).drop("valor_provision_limpio")

# 3. Crear columna de fecha normalizada (solo fecha, sin hora)
print("  3. Normalizar columna de fecha")
df_silver = df_silver.withColumn(
    "fecha_analisis",
    to_date(col("fecha"))
)

# 4. Crear categoría de valor de inventario
print("  4. Crear categorías de análisis")
df_silver = df_silver.withColumn(
    "rango_valor_inventario",
    when(col("valor_real_cop_dolares") < 10000000, "Bajo")
    .when(col("valor_real_cop_dolares") < 100000000, "Medio")
    .when(col("valor_real_cop_dolares") < 500000000, "Alto")
    .otherwise("Muy Alto")
)

# 5. Filtrar registros con valores críticos nulos
print("  5. Filtrar registros con valores nulos críticos")
registros_antes = df_silver.count()
df_silver = df_silver.filter(
    col("item").isNotNull() &
    col("unidad_de_negocio").isNotNull() &
    col("fecha").isNotNull() &
    col("valor_real_cop_dolares").isNotNull()
)
registros_despues = df_silver.count()
print(f"    - Registros antes: {registros_antes:,}")
print(f"    - Registros después: {registros_despues:,}")
print(f"    - Registros eliminados: {registros_antes - registros_despues:,}")

# 6. Guardar tabla Silver
print(f"\n💾 Guardando tabla SILVER...")
tabla_silver = "workshop_aleja.default.inventario_silver"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_silver)

print(f"\n✅ Tabla SILVER creada exitosamente: {tabla_silver}")
print(f"📊 Registros finales: {df_silver.count():,}")
print(f"📊 Columnas totales: {len(df_silver.columns)}")

⚪ Creando tabla SILVER...

⚙️ Aplicando transformaciones...

  1. Convertir 'saldo' a numérico
  2. Convertir 'valor_provision' a numérico
  3. Normalizar columna de fecha
  4. Crear categorías de análisis
  5. Filtrar registros con valores nulos críticos
    - Registros antes: 22,071
    - Registros después: 22,071
    - Registros eliminados: 0

💾 Guardando tabla SILVER...

✅ Tabla SILVER creada exitosamente: workshop_aleja.default.inventario_silver
📊 Registros finales: 22,071
📊 Columnas totales: 38


In [0]:
# Verificar calidad de datos en Silver
print("🔍 VERIFICACIÓN DE CALIDAD - Tabla SILVER\n")

df_check = spark.table("workshop_aleja.default.inventario_silver")

# Conteo por unidad de negocio
print("1. Registros por Unidad de Negocio:")
df_check.groupBy("unidad_de_negocio").count().orderBy("count", ascending=False).show()

# Conteo por categoría de provisión
print("\n2. Registros por Categoría de Provisión:")
df_check.groupBy("categoria_provision_de_inventario").count().orderBy("count", ascending=False).show()

# Verificar valores nulos en columnas clave
print("\n3. Valores nulos en columnas clave:")
df_check.select(
    [col(c).isNull().cast("int").alias(c) for c in [
        "item", "saldo_numerico", "valor_real_cop_dolares", 
        "valor_provision_numerico", "categoria_provision_de_inventario"
    ]]
).select(
    [sum(col(c)).alias(f"{c}_nulls") for c in [
        "item", "saldo_numerico", "valor_real_cop_dolares", 
        "valor_provision_numerico", "categoria_provision_de_inventario"
    ]]
).show()

# Muestra de datos transformados
print("\n4. Muestra de datos transformados:")
df_check.select(
    "item", "descripcion_item", "unidad_de_negocio", 
    "saldo_numerico", "valor_real_cop_dolares", "valor_provision_numerico",
    "categoria_provision_de_inventario", "rango_valor_inventario"
).limit(5).show(truncate=False)

## 🟡 PASO 4: Capa GOLD - Métricas de Negocio

La capa Gold contiene datos altamente agregados y optimizados para análisis de negocio y dashboards.

**Tablas Gold a crear:**
1. **`inventario_gold_unidad_negocio`**: Métricas por unidad de negocio
2. **`inventario_gold_producto`**: Métricas por producto (item)
3. **`inventario_gold_categoria_provision`**: Métricas por categoría de provisión
4. **`inventario_gold_temporal`**: Serie temporal para análisis de tendencias
5. **`vw_dashboard_inventario`**: Vista consolidada para dashboards

**Métricas clave:**
* Valor total de inventario
* Provisión de inventario
* Stock actual por dimensión
* Análisis temporal

In [0]:
%sql
-- GOLD 1: Métricas agregadas por Unidad de Negocio
-- Proporciona vista consolidada del inventario por cada unidad de negocio

CREATE OR REPLACE TABLE workshop_aleja.default.inventario_gold_unidad_negocio AS
SELECT 
    unidad_de_negocio,
    COUNT(DISTINCT item) as total_productos,
    SUM(saldo_numerico) as cantidad_total_stock,
    SUM(valor_real_cop_dolares) as valor_total_inventario,
    SUM(valor_provision_numerico) as valor_total_provision,
    AVG(valor_real_cop_dolares) as valor_promedio_producto,
    MAX(fecha_analisis) as ultima_actualizacion
FROM workshop_aleja.default.inventario_silver
GROUP BY unidad_de_negocio
ORDER BY valor_total_inventario DESC;

-- Mostrar resultados
SELECT 
    unidad_de_negocio,
    total_productos,
    FORMAT_NUMBER(cantidad_total_stock, 0) as stock_total,
    CONCAT('$', FORMAT_NUMBER(valor_total_inventario, 2)) as valor_inventario,
    CONCAT('$', FORMAT_NUMBER(valor_total_provision, 2)) as provision_total,
    ultima_actualizacion
FROM workshop_aleja.default.inventario_gold_unidad_negocio;

unidad_de_negocio,total_productos,stock_total,valor_inventario,provision_total,ultima_actualizacion
TL,546,"14,712,128","$73,276,812,762.38","$15,662,312,830.10",2026-04-30
DT,237,"4,664,942","$72,052,106,374.30","$12,313,923,752.70",2026-04-30
PR,72,"4,064,411","$52,180,652,525.84","$6,913,081,657.20",2026-04-30
TR,319,"258,707","$13,056,402,388.61","$1,829,821,903.60",2026-04-30
NB,22,604,"$807,626,595.10","$222,009,876.00",2026-04-30


In [0]:
%sql
-- GOLD 2: Métricas por Producto (Item)
-- Análisis detallado de cada producto: stock, valor, provisión

CREATE OR REPLACE TABLE workshop_aleja.default.inventario_gold_producto AS
SELECT 
    item,
    descripcion_item,
    tipo_de_item,
    grupo_inventario,
    SUM(saldo_numerico) as stock_actual,
    SUM(valor_real_cop_dolares) as valor_total,
    SUM(valor_provision_numerico) as provision_total,
    AVG(valor_real_cop_dolares / NULLIF(saldo_numerico, 0)) as valor_unitario_promedio,
    COUNT(DISTINCT unidad_de_negocio) as unidades_negocio_con_stock,
    MAX(categoria_provision_de_inventario) as categoria_provision,
    MAX(fecha_analisis) as ultima_actualizacion
FROM workshop_aleja.default.inventario_silver
WHERE saldo_numerico > 0
GROUP BY item, descripcion_item, tipo_de_item, grupo_inventario
ORDER BY valor_total DESC;

-- Mostrar Top 20 productos por valor
SELECT 
    item,
    descripcion_item,
    tipo_de_item,
    FORMAT_NUMBER(stock_actual, 2) as stock,
    CONCAT('$', FORMAT_NUMBER(valor_total, 2)) as valor_total,
    CONCAT('$', FORMAT_NUMBER(provision_total, 2)) as provision,
    categoria_provision
FROM workshop_aleja.default.inventario_gold_producto
LIMIT 20;

item,descripcion_item,tipo_de_item,stock,valor_total,provision,categoria_provision
ECR-35,Producto ECR-35,Acabados,"204,852.00","$10,015,951,532.95",$0.00,Regular
00030121,Producto 00030121,Materia Prima,"639,386.45","$9,606,259,846.34","$480,489,819.00",Regular
ECR-15N,Producto ECR-15N,Acabados,"510,288.00","$7,666,971,090.76",$0.00,Regular
015100-PLPBR,Producto 015100-PLPBR,Materia Prima,"414,852.40","$7,080,152,097.63","$1,697,655,173.60",Regular
PG-5750E,Producto PG-5750E,Acabados,"414,186.00","$6,774,978,549.60","$1,322,714,512.50",Regular
ECR-35-COL,Producto ECR-35-COL,Acabados,"118,931.00","$6,319,250,887.26","$5,152,274,175.50",Regular
IP-101N,Producto IP-101N,Acabados,"560,209.00","$6,280,715,840.97",$0.00,Regular
SIPA-83,Producto SIPA-83,Acabados,"1,588,926.00","$5,968,350,429.85","$107,043,194.70",Regular
SIPA-84,Producto SIPA-84,Acabados,"917,443.00","$5,011,897,493.22",$0.00,Regular
015091,Producto 015091,Materia Prima,"377,609.61","$4,395,531,378.43","$255,021,482.50",Regular


In [0]:
%sql
-- GOLD 3: Métricas por Categoría de Provisión de Inventario
-- Análisis de la distribución del inventario por categoría de provisión

CREATE OR REPLACE TABLE workshop_aleja.default.inventario_gold_categoria_provision AS
SELECT 
    categoria_provision_de_inventario,
    COUNT(DISTINCT item) as total_productos,
    SUM(saldo_numerico) as cantidad_total,
    SUM(valor_real_cop_dolares) as valor_total_inventario,
    SUM(valor_provision_numerico) as valor_total_provision,
    ROUND(SUM(valor_real_cop_dolares) / SUM(SUM(valor_real_cop_dolares)) OVER () * 100, 2) as porcentaje_valor,
    MAX(fecha_analisis) as ultima_actualizacion
FROM workshop_aleja.default.inventario_silver
GROUP BY categoria_provision_de_inventario
ORDER BY valor_total_inventario DESC;

-- Mostrar distribución completa
SELECT 
    categoria_provision_de_inventario,
    total_productos,
    FORMAT_NUMBER(cantidad_total, 0) as cantidad,
    CONCAT('$', FORMAT_NUMBER(valor_total_inventario, 2)) as valor_inventario,
    CONCAT('$', FORMAT_NUMBER(valor_total_provision, 2)) as provision,
    CONCAT(porcentaje_valor, '%') as pct_del_total
FROM workshop_aleja.default.inventario_gold_categoria_provision;

In [0]:
%sql
-- GOLD 4: Serie Temporal para Análisis de Tendencias
-- Permite análisis temporal del inventario por fecha, unidad de negocio y categoría

CREATE OR REPLACE TABLE workshop_aleja.default.inventario_gold_temporal AS
SELECT 
    fecha_analisis,
    unidad_de_negocio,
    categoria_provision_de_inventario,
    COUNT(DISTINCT item) as productos_unicos,
    SUM(saldo_numerico) as cantidad_total,
    SUM(valor_real_cop_dolares) as valor_total_inventario,
    SUM(valor_provision_numerico) as valor_total_provision
FROM workshop_aleja.default.inventario_silver
GROUP BY fecha_analisis, unidad_de_negocio, categoria_provision_de_inventario
ORDER BY fecha_analisis DESC, valor_total_inventario DESC;

-- Mostrar resumen por fecha y unidad de negocio
SELECT 
    fecha_analisis,
    unidad_de_negocio,
    SUM(productos_unicos) as total_productos,
    CONCAT('$', FORMAT_NUMBER(SUM(valor_total_inventario), 2)) as valor_total
FROM workshop_aleja.default.inventario_gold_temporal
GROUP BY fecha_analisis, unidad_de_negocio
ORDER BY fecha_analisis DESC, valor_total DESC
LIMIT 50;

In [0]:
%sql
-- VISTA CONSOLIDADA PARA DASHBOARD
-- Combina datos de Silver y Gold para análisis completo
-- Esta vista será la fuente principal para dashboards y Power BI

CREATE OR REPLACE VIEW workshop_aleja.default.vw_dashboard_inventario AS
SELECT 
    s.item,
    s.descripcion_item,
    s.unidad_de_negocio,
    s.tipo_de_item,
    s.grupo_inventario,
    s.categoria_provision_de_inventario,
    s.saldo_numerico as stock,
    s.valor_real_cop_dolares as valor_inventario,
    s.valor_provision_numerico as valor_provision,
    s.rango_valor_inventario,
    s.fecha_analisis,
    p.valor_unitario_promedio,
    p.unidades_negocio_con_stock,
    u.total_productos as productos_en_unidad,
    u.valor_total_inventario as valor_total_unidad,
    c.porcentaje_valor as porcentaje_categoria
FROM workshop_aleja.default.inventario_silver s
LEFT JOIN workshop_aleja.default.inventario_gold_producto p 
    ON s.item = p.item
LEFT JOIN workshop_aleja.default.inventario_gold_unidad_negocio u 
    ON s.unidad_de_negocio = u.unidad_de_negocio
LEFT JOIN workshop_aleja.default.inventario_gold_categoria_provision c
    ON s.categoria_provision_de_inventario = c.categoria_provision_de_inventario
WHERE s.valor_real_cop_dolares > 0;

-- Verificar vista con muestra de datos
SELECT 
    item,
    descripcion_item,
    unidad_de_negocio,
    categoria_provision_de_inventario,
    FORMAT_NUMBER(stock, 2) as stock,
    CONCAT('$', FORMAT_NUMBER(valor_inventario, 2)) as valor,
    rango_valor_inventario
FROM workshop_aleja.default.vw_dashboard_inventario
LIMIT 100;

## ✅ PIPELINE MEDALLION COMPLETADO

### Tablas Creadas:

#### 🟫 **BRONZE** - Datos Crudos
* `workshop_aleja.default.inventario_bronze`
* 22,071 registros, 34 columnas
* Datos sin transformar + metadatos de auditoría

#### ⚪ **SILVER** - Datos Limpios
* `workshop_aleja.default.inventario_silver`
* 22,071 registros, 38 columnas
* Conversiones numéricas, validaciones, columnas calculadas

#### 🟡 **GOLD** - Métricas de Negocio
1. `workshop_aleja.default.inventario_gold_unidad_negocio` - Agregación por unidad de negocio
2. `workshop_aleja.default.inventario_gold_producto` - Agregación por producto
3. `workshop_aleja.default.inventario_gold_categoria_provision` - Agregación por categoría
4. `workshop_aleja.default.inventario_gold_temporal` - Serie temporal
5. `workshop_aleja.default.vw_dashboard_inventario` - Vista consolidada

---

## 📊 PRÓXIMOS PASOS

### 1. Dashboard en Databricks
* Ve a **Dashboards** → **Create Dashboard**
* Usa las tablas Gold y la vista `vw_dashboard_inventario`
* Crea visualizaciones: KPIs, gráficos de barras, series temporales

### 2. Conexión a Power BI

**Método Rápido - Partner Connect:**
1. **Marketplace** → Busca **Power BI**
2. Selecciona tu SQL Warehouse
3. **Download connection file** (.pbids)
4. Abre en Power BI Desktop
5. Ingresa credenciales (Personal Access Token o OAuth)
6. Selecciona las tablas Gold

**Método Manual:**
1. **SQL Warehouses** → Copia **Server Hostname** y **HTTP Path**
2. Power BI: **Obtener datos** → **Databricks**
3. Pega credenciales
4. Elige **DirectQuery** (tiempo real) o **Import** (mejor rendimiento)
5. Selecciona tablas Gold

**Generar Personal Access Token:**
* Perfil → **Settings** → **Developer** → **Manage tokens**
* **Generate new token** → Nombre "Power BI" → Lifetime 90 días
* ⚠️ Copia inmediatamente (solo se muestra una vez)

---

## 📋 TABLAS RECOMENDADAS PARA POWER BI

Conecta estas tablas/vistas en Power BI:
* ✅ `vw_dashboard_inventario` (vista principal consolidada)
* ✅ `inventario_gold_unidad_negocio`
* ✅ `inventario_gold_producto`
* ✅ `inventario_gold_categoria_provision`
* ✅ `inventario_gold_temporal`

---

## 🎯 MÉTRICAS CLAVE PARA DASHBOARDS

**KPIs Principales:**
* Valor Total de Inventario
* Valor Total de Provisión
* Cantidad Total de Productos
* Stock Total

**Dimensiones de Análisis:**
* Unidad de Negocio
* Categoría de Provisión
* Tipo de Ítem
* Rango de Valor
* Fecha (temporal)

**Visualizaciones Sugeridas:**
* 📊 Gráfico de barras: Valor por Unidad de Negocio
* 🥧 Gráfico circular: Distribución por Categoría de Provisión
* 📈 Línea temporal: Evolución del Valor de Inventario
* 📋 Tabla: Top Productos por Valor
* 🗺️ Heatmap: Provisión por Unidad y Categoría